<a href="https://colab.research.google.com/github/Shahla-AAFS/Nail_disorder_detection/blob/main/Nail_disorder_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [2]:
# FIX — Drive is already mounted. Just run this cell to confirm and continue.
# =============================================================================

import os

# Drive is already connected — just verify the path exists
print("Drive status: already mounted")
print()

# Check your nail_split folder
base_path = "/content/drive/MyDrive/nail_split"

if os.path.exists(base_path):
    print(f"base_path found: {base_path}")

    train_dir = os.path.join(base_path, "train")
    val_dir   = os.path.join(base_path, "val")

    print(f"train/ exists : {os.path.exists(train_dir)}")
    print(f"val/   exists : {os.path.exists(val_dir)}")

    if os.path.exists(train_dir):
        print(f"\nTrain classes : {os.listdir(train_dir)}")
    if os.path.exists(val_dir):
        print(f"Val   classes : {os.listdir(val_dir)}")

    print("\nPath is correct. Proceed to the CFG + generator cells.")

else:
    # Drive is mounted but nail_split folder not found at expected location
    print(f"nail_split NOT found at: {base_path}")
    print("\nContents of MyDrive root:")
    for item in sorted(os.listdir("/content/drive/MyDrive/")):
        print(f"  {item}")
    print("\nFind your folder name above and update base_path accordingly.")

Drive status: already mounted

nail_split NOT found at: /content/drive/MyDrive/nail_split

Contents of MyDrive root:
  nail_models

Find your folder name above and update base_path accordingly.


In [3]:
!ls "/content/drive/MyDrive/main nail dataset"

ls: cannot access '/content/drive/MyDrive/main nail dataset': No such file or directory


In [4]:
dataset_path = "/content/drive/MyDrive/main nail dataset"

In [6]:
import os
print(os.listdir("/content/drive/MyDrive/main nail dataset"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/main nail dataset'

In [ ]:
!pip install split-folders

In [ ]:
import splitfolders

splitfolders.ratio(
    "/content/drive/MyDrive/main nail dataset",
    output="/content/nail_split",
    seed=42,
    ratio=(0.8, 0.2)   # 80% train, 20% validation
)

In [ ]:
!ls /content/nail_split

In [ ]:
!ls /content/nail_split/train

In [ ]:
!ls /content/nail_split/val

In [ ]:
!ls /content/drive/MyDrive/nail_split

In [ ]:
!ls -lh /content/drive/MyDrive/

In [ ]:
!cp -rv /content/nail_split /content/drive/MyDrive/

In [ ]:
#check if the splitted folders are exist inside drive
import os

base_path = "/content/drive/MyDrive/nail_split"
print("Train classes:", os.listdir(os.path.join(base_path, "train")))
print("Validation classes:", os.listdir(os.path.join(base_path, "val")))

In [ ]:
#Install  import all libraries

import os, gc, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score
)

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU'))

In [ ]:
# ── CELL S2 ── Global config
# ───────────────────────────
CFG = {
    "base_path"      : "/content/drive/MyDrive/nail_split",
    "save_dir"       : "/content/drive/MyDrive/nail_models",
    "img_size"       : (224, 224),
    "batch_size"     : 16,
    "num_classes"    : 6,
    "seed"           : 42,
    "phase1_epochs"  : 10,
    "phase2_epochs"  : 20,
    "phase1_lr"      : 1e-3,
    "phase2_lr"      : 1e-5,
    "unfreeze_layers": 30,
    "dropout_rate"   : 0.4,
    "dense_units"    : 256,
    "class_names"    : [
        'Acral_Lentiginous_Melanoma',
        'Healthy_Nail',
        'Onychogryphosis',
        'blue_finger',
        'clubbing',
        'pitting'
    ]
}

TRAIN_DIR = os.path.join(CFG["base_path"], "train")
VAL_DIR   = os.path.join(CFG["base_path"], "val")
os.makedirs(CFG["save_dir"], exist_ok=True)

print("Train dir :", TRAIN_DIR)
print("Val   dir :", VAL_DIR)
print("Save  dir :", CFG["save_dir"])


In [ ]:
# ── CELL S3 ── Build shared data generators (run ONCE, reused by all 3 models)
# ──────────────────────────────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale            = 1.0 / 255,
    horizontal_flip    = True,
    rotation_range     = 15,
    zoom_range         = 0.2,
    width_shift_range  = 0.1,
    height_shift_range = 0.1,
    brightness_range   = [0.8, 1.2],
    fill_mode          = 'nearest'
)
val_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size = CFG["img_size"],
    batch_size  = CFG["batch_size"],
    class_mode  = 'categorical',
    shuffle     = True,
    seed        = CFG["seed"]
)
val_gen = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size = CFG["img_size"],
    batch_size  = CFG["batch_size"],
    class_mode  = 'categorical',
    shuffle     = False
)

print("Class index mapping :", train_gen.class_indices)
print("Train batches       :", len(train_gen))
print("Val   batches       :", len(val_gen))